# 05 — jazz_collab: el efecto en una red social, no biológica

**Pregunta que se probó:** ¿Aparece la anti-centralidad grado↔V en una red de
colaboración musical (jazz_collab, Netzschleuder), con la misma fuerza que en
redes biológicas?

**Resultado:** SÍ. Spearman(grado, V) ≈ −0.998 en jazz_collab, comparable a
los conectomas. Este notebook fue el punto donde se confirmó que el efecto no
está limitado a sistemas biológicos — de ahí el nombre original
"descubrimiento jazz" en el desarrollo del proyecto (aunque, como se documenta
en el paper, el objeto V mismo no resultó ser nuevo — Sección 4).

Ver detalle completo en `paper/SPG_final_v9.docx`, Sección 5.2.


## Fuentes de datos

- **Jazz collaboration network**
  https://networks.skewed.de/net/jazz_collab
- **C. elegans metabolic**
  https://networks.skewed.de/net/celegans_metabolic
- **Food web (Little Rock)**
  https://networks.skewed.de/net/foodweb_little_rock
- **Bitcoin alpha**
  https://networks.skewed.de/net/bitcoin_alpha
- **Fly larva**
  https://networks.skewed.de/net/fly_larva
- **Interactome PDZ**
  https://networks.skewed.de/net/interactome_pdz
- **Interactome Stelzl**
  https://networks.skewed.de/net/interactome_stelzl
- **Yeast transcription**
  https://networks.skewed.de/net/yeast_transcription
- **Interactome Vidal** — enlace pendiente de confirmar (no se recibió URL)


> **Nota:** de cada dataset se usó únicamente el archivo de tipo `edges` (lista de aristas). No se usaron archivos de nodos ni de metadatos adicionales de Netzschleuder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Experimental'
print("¿existe la carpeta?:", os.path.isdir(CARPETA))
if os.path.isdir(CARPETA):
    print("archivos que veo:")
    for f in sorted(os.listdir(CARPETA)):
        print("  ", f)

¿existe la carpeta?: True
archivos que veo:
   fly_larva.csv
   foodweb_little_rock.csv
   interactome_pdz.csv
   interactome_stelzl.csv
   interactome_vidal.csv
   jazz_collab.csv
   nteractome_figeys.csv
   yeast_transcription.csv


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N

TABLA = {
    'LCI': +0.267, 'BPmaps': -0.032, 'Microarray': -1.000,
    'Phenotypes': -0.999, 'IntegratedNetwork': -1.000, 'Interolog': -0.190,
    'Genetic': -0.632, 'WI8': -0.602, 'wi2004': -0.507, 'wi2007': -0.421,
}

def analizar(archivo):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['source','target','eid','weight'])
    G = nx.Graph(); G.add_edges_from(df[['source','target']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    return A.shape[0], spearmanr(s.degree,s.V)[0], spearmanr(s.degree,s.tau_tilde)[0]

print(f"{'red':20s} {'N':>5s} {'Sp(k,V)':>9s} {'Sp(k,tt)':>9s} {'tabla':>8s} {'match':>6s}")
print("-"*62)
for f in sorted(os.listdir('.')):
    if f.endswith('.csv') and f not in ('edges.csv','nodes.csv','gprops.csv'):
        red = f[:-4]
        try:
            N, rv, rtt = analizar(f)
            esp = TABLA.get(red)
            match = "OK" if (esp is not None and abs(rtt-esp)<0.15) else ("?" if esp is None else "NO")
            esp_s = f"{esp:+.3f}" if esp is not None else "nuevo"
            print(f"{red:20s} {N:5d} {rv:+.3f}   {rtt:+.3f}   {esp_s:>8s} {match:>6s}")
        except Exception as e:
            print(f"{red:20s} ERROR: {str(e)[:30]}")


red                      N   Sp(k,V)  Sp(k,tt)    tabla  match
--------------------------------------------------------------


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N; self.lambda2 = ev[k0]
        self.n_zero = int((np.abs(ev)<1e-6).sum())   # chequeo disconexión

def medir(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None,
                     names=['s','t','e','w'])
    G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    N = s.N; deg = s.degree
    rv  = spearmanr(deg, s.V)[0]
    rtt = spearmanr(deg, s.tau_tilde)[0]
    dens = A.sum()/(N*(N-1)); cv = deg.std()/deg.mean()
    # TEST DE CONFIABILIDAD automático
    fiable = "SI" if np.sign(rv)==np.sign(rtt) else "NO-descartar"
    print(f"{nombre:24s} N={N:5d} dens={dens:.3f} CV={cv:.2f} "
          f"Sp(V)={rv:+.3f} Sp(tt)={rtt:+.3f}  [{fiable}]")
    return dict(red=nombre, N=N, dens=dens, cv=cv, sp_V=rv, sp_tt=rtt, fiable=fiable)


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N; self.lambda2 = ev[k0]

def medir(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None)
    # tomar solo las 2 primeras columnas (source, target) sin importar cuántas haya
    df = df.iloc[:, :2]; df.columns = ['s','t']
    G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    N = s.N; deg = s.degree
    rv  = spearmanr(deg, s.V)[0]
    rtt = spearmanr(deg, s.tau_tilde)[0]
    dens = A.sum()/(N*(N-1)); cv = deg.std()/deg.mean()
    fiable = "SI" if np.sign(rv)==np.sign(rtt) else "NO-descartar"
    print(f"{nombre:24s} {N:5d} {dens:6.3f} {cv:5.2f} {rv:+7.3f} {rtt:+7.3f} {fiable:>12s}")
    return dict(red=nombre, N=N, dens=dens, cv=cv, sp_V=rv, sp_tt=rtt, fiable=fiable)

print("cargado OK — ahora corre el bloque de medir")


cargado OK — ahora corre el bloque de medir


In [ ]:
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Experimental'  # ajusta si tu ruta es otra
import os

# primero confirmamos QUÉ archivos ve y la ruta correcta
print("archivos .csv en la carpeta:")
for f in sorted(os.listdir(CARPETA)):
    if f.endswith('.csv'):
        print("  ", f)
print("="*60)

resultados = []
print(f"{'red':24s} {'N':>5s} {'dens':>6s} {'CV':>5s} {'Sp(V)':>7s} {'Sp(tt)':>7s} {'fiable':>12s}")
print("-"*72)
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    try:
        resultados.append(medir(os.path.join(CARPETA, f), f[:-4]))
    except Exception as e:
        print(f"{f[:-4]:24s} ERROR: {str(e)[:40]}")


archivos .csv en la carpeta:
   fly_larva.csv
   foodweb_little_rock.csv
   interactome_pdz.csv
   interactome_stelzl.csv
   interactome_vidal.csv
   jazz_collab.csv
   nteractome_figeys.csv
   yeast_transcription.csv
red                          N   dens    CV   Sp(V)  Sp(tt)       fiable
------------------------------------------------------------------------
fly_larva                 2952  0.022  0.66  -1.000  -0.851           SI
foodweb_little_rock        183  0.146  0.78  -0.996  -0.926           SI
interactome_pdz            161  0.016  1.12  -0.680  +0.147 NO-descartar
interactome_stelzl        1615  0.002  1.85  -0.861  -0.444           SI
interactome_vidal         2783  0.002  1.63  -0.911  -0.588           SI
jazz_collab                198  0.141  0.63  -0.998  -0.658           SI
nteractome_figeys         2217  0.003  2.95  -0.934  -0.802           SI
yeast_transcription        664  0.005  1.79  -0.840  -0.120           SI


In [ ]:
import networkx as nx, numpy as np, pandas as pd, os
from scipy.stats import spearmanr, pearsonr

def comparar_dominio(archivo, nombre, dominio):
    df = pd.read_csv(archivo, comment='#', header=None).iloc[:,:2]
    df.columns=['s','t']
    G = nx.Graph(); G.add_edges_from(df.values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); nodes=list(G.nodes())
    # L+ii directo (la definición canónica, sin pasar por SPG)
    L = np.diag(A.sum(1)) - A
    Lplus_diag = np.diag(np.linalg.pinv(L))
    deg = np.array([G.degree(n) for n in nodes])
    # centralidades clásicas
    cfc = np.array([nx.current_flow_closeness_centrality(G)[n] for n in nodes])
    clo = np.array([nx.closeness_centrality(G)[n] for n in nodes])
    print(f"\n=== {nombre} [{dominio}] N={len(nodes)} ===")
    print(f"  Spearman(grado, L+ii)        = {spearmanr(deg, Lplus_diag)[0]:+.3f}")
    print(f"  Spearman(L+ii, current_flow) = {spearmanr(Lplus_diag, cfc)[0]:+.3f}")
    print(f"  Spearman(L+ii, closeness)    = {spearmanr(Lplus_diag, clo)[0]:+.3f}")
    print(f"  Pearson (L+ii, current_flow) = {pearsonr(Lplus_diag, cfc)[0]:+.3f}")

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/HumanBrain'
# biológicas y NO biológicas, mezcladas a propósito
casos = [
    ('celegans_metabolic.csv','celegans_metabolic','BIOLÓGICO'),
    ('foodweb_little_rock.csv','foodweb','BIOLÓGICO'),
    ('jazz_collab.csv','jazz','SOCIAL - no bio'),
    ('bitcoin_alpha.csv','bitcoin','FINANCIERO - no bio'),
]
for arch, nom, dom in casos:
    try: comparar_dominio(os.path.join(CARPETA,arch), nom, dom)
    except Exception as e: print(f"{nom}: ERROR {str(e)[:40]}")


celegans_metabolic: ERROR [Errno 2] No such file or directory: '/c
foodweb: ERROR [Errno 2] No such file or directory: '/c
jazz: ERROR [Errno 2] No such file or directory: '/c
bitcoin: ERROR [Errno 2] No such file or directory: '/c
